# Joint space-time realignment — recovery demo

**What this notebook shows.** Slice timing *and* head motion jointly corrupt an
fMRI series. `ffs_nwarp` (`processing/spacetime.py`, after Roche 2011) can undo
both in a **single** resample by folding the slice-timing shift into the motion
warp. Here we build a synthetic series whose ground truth is known exactly, and
measure how well that single fix recovers it against two references:

| method | what it is |
|---|---|
| **joint (following)** | `ffs_nwarp -tpattern -tfollow` — samples each temporal tap at *its own* frame's pose (tissue-following) |
| **joint (frozen)** | `ffs_nwarp -tpattern` — motion + slice timing in one interpolation, one pose per output frame (slow-motion assumption) |
| **tshift → moco** | `ffs_slicetime` (static 3dTshift) *then* `ffs_nwarp` motion-only — the correct two-step |
| **motion-only** | `ffs_nwarp` with no slice-timing correction |

The **frozen** joint and **tshift** agree in most regimes; they diverge from
**following** when motion sweeps tissue between scanner locations within the
temporal window — set `motion_mode="oscillate"` (optionally `sharp_edge=True`) to
see the brain-edge case where following wins decisively.

**The physics we encode.** A continuous world
$I(x,\tau) = \mathrm{struct}(x)\,\bigl(1 + a\sin(2\pi\,\nu(x_k)\,\tau + \phi)\bigr)$:

- $\mathrm{struct}(x)$ — spatial structure (a gentle checkerboard under a smooth
  envelope, or a real image you drop in) so motion is visible.
- $\nu(x_k)$ — the temporal frequency is a property of the **tissue**, shared
  across blocks of a few neighbouring slices (a single-slice time course is
  unphysical), ramping slow → near-Nyquist.
- $\Delta(k)$ — the acquisition offset is a property of the **scanner slice**
  (multiband, interleaved: the realistic worst case for motion).

Forward corruption is evaluated **analytically** (look up the tissue at the
motion-mapped coordinate, sample the world at the scanner slice's acquisition
time), so there is *no inverse crime*: the only errors `ffs_nwarp` can be blamed
for are its own interpolation. Optional measurement **noise** (the repo's
`generate_fmri_noise`) can be added at acquisition to watch how gracefully each
method degrades.

Everything is driven by the **config cell** below. Scale the motion, flip to
through-plane, turn noise on, or drop in a real image / real motion params.

In [ ]:
import os, tempfile
import numpy as np
import matplotlib.pyplot as plt
import torch
import nibabel as nib

from fastfuncstuff.cli.nwarp import main as nwarp_main
from fastfuncstuff.cli.slicetime import main as slicetime_main
from fastfuncstuff.simulation.noise import generate_fmri_noise

plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True,
                     "axes.axisbelow": True, "grid.alpha": 0.25})
np.set_printoptions(precision=3, suppress=True)

## Configuration — edit me

Everything below reads from `CFG`. The drop-in hooks (`real_image_path`,
`real_motion_path`, `real_slicetiming`) are `None` by default; set them to use
your own data with the synthetic signal layered on top.

In [ ]:
CFG = dict(
    # ---- grid & timing -----------------------------------------------------
    shape=(48, 48, 20),      # (nx, ny, nz)
    nframes=64,
    tr=1.0,                  # seconds
    multiband=2,             # simultaneously-excited slices
    slice_order="interleaved",   # "interleaved" | "ascending"

    # ---- signal ------------------------------------------------------------
    slice_block=3,           # neighbouring slices sharing one time course
    f_lo=0.02, f_hi=0.45,    # slice-block frequency ramp (Hz); Nyquist = 0.5/tr
    signal_amp=0.5,          # modulation depth on top of the spatial structure
    checker_contrast=0.35,   # spatial checker depth (turn up for a punchier image)
    checker_period=12.0,

    # ---- motion ------------------------------------------------------------
    # "oscillate" (default here) showcases tissue-following; flip to "drift" for the
    # slow, realistic validation where all slice-timing-aware methods agree.
    motion_mode="oscillate", # "drift" (slow, realistic) | "oscillate" (fast edge sweep)
    through_plane=False,     # drift mode: False = in-plane only; True = harder
    motion_scale=1.0,        # multiply ALL motion (real-but-2x => 2.0)
    osc_amp=4.0,             # oscillate mode: +/- this many vox in x, every frame
    sharp_edge=True,         # sharp brain/air disk (dramatic under oscillate)
    motion_seed=1,
    amp_deg=2.0,             # base rotation amplitude (deg) before motion_scale
    amp_trans=1.5,           # base translation amplitude (vox) before motion_scale

    # ---- noise (repo synthetic engine; added at acquisition) ---------------
    noise_enabled=False,
    noise_sd=0.05,           # SD in signal units (signal ranges ~0.3..0.9)
    noise_resp_strength=3.0, # physiological knobs passed to generate_fmri_noise
    noise_cardiac_strength=5.0,
    noise_seed=0,

    # ---- resampling --------------------------------------------------------
    interp="wsinc5",         # spatial kernel
    tinterp="wsinc5",        # temporal kernel for the joint path
    device="cpu",

    # ---- drop-in real data (optional) --------------------------------------
    real_image_path=None,    # 3D NIfTI cropped to ~CFG['shape'] -> replaces struct()
    real_motion_path=None,   # multi-row 12-col aff12 (voxel frame) -> replaces make_motion
    real_slicetiming=None,   # list/array of per-slice offsets (s) -> replaces make_slice_times
)
NYQUIST = 0.5 / CFG["tr"]
print(f"Nyquist = {NYQUIST:.3f} Hz   frames = {CFG['nframes']}   grid = {CFG['shape']}")

## World model — spatial structure and per-slice signal

In [ ]:
def make_struct_analytic(shape, checker_period, checker_contrast, margin=0.1):
    """Callable struct(coords): gentle checkerboard * smooth in-plane disk."""
    nx, ny, nz = shape
    cx, cy = (nx - 1) / 2, (ny - 1) / 2
    rx, ry = nx * (0.5 - margin), ny * (0.5 - margin)

    def struct(coords):
        xi, xj = coords[0], coords[1]
        checker = 1.0 + checker_contrast * np.cos(2 * np.pi * xi / checker_period) \
            * np.cos(2 * np.pi * xj / checker_period)
        r = np.sqrt(((xi - cx) / rx) ** 2 + ((xj - cy) / ry) ** 2)
        env = 0.5 * (1.0 + np.cos(np.pi * np.clip(r, 0.0, 1.0)))  # C1, no cusp
        return checker * env
    return struct


def make_struct_from_image(vol):
    """Callable struct(coords) that samples a real 3D volume (drop-in)."""
    from scipy.ndimage import map_coordinates
    vol = np.asarray(vol, np.float32)
    vol = vol / (np.percentile(vol, 99) + 1e-8)  # normalise to ~[0,1]

    def struct(coords):
        return map_coordinates(vol, coords[:3], order=1, mode="nearest")
    return struct


def make_struct_sharp(shape, checker_contrast, checker_period, R_frac=0.36, taper=2.0):
    """Disk with a sharp-ish brain/air edge (smoothstep). The edge going in and out
    of a voxel under fast motion is the case that separates tissue-following."""
    nx, ny, nz = shape
    cx, cy, R = (nx - 1) / 2, (ny - 1) / 2, min(nx, ny) * R_frac

    def struct(coords):
        xi, xj = coords[0], coords[1]
        r = np.sqrt((xi - cx) ** 2 + (xj - cy) ** 2)
        t = np.clip((R - r) / taper, 0, 1)
        checker = 1.0 + checker_contrast * np.cos(2 * np.pi * xi / checker_period) \
            * np.cos(2 * np.pi * xj / checker_period)
        return checker * (t * t * (3 - 2 * t))
    return struct


def make_temporal(shape, slice_block, f_lo, f_hi, amp, seed=0):
    """Per-slice (nu, phi) shared across blocks of `slice_block` slices."""
    nz = shape[2]
    rng = np.random.default_rng(seed)
    n_blocks = int(np.ceil(nz / slice_block))
    bnu = np.linspace(f_lo, f_hi, n_blocks)
    bph = rng.uniform(0, 2 * np.pi, n_blocks)
    blk = np.minimum(np.arange(nz) // slice_block, n_blocks - 1)
    nu, phi = bnu[blk], bph[blk]

    def modulation(coords, tau):
        k0 = np.clip(np.round(coords[2]).astype(int), 0, nz - 1)
        return 1.0 + amp * np.sin(2 * np.pi * nu[k0] * tau + phi[k0])
    return nu, phi, modulation


# Build struct: real image if provided, else analytic (smooth or sharp-edged).
if CFG["real_image_path"]:
    _img = np.asarray(nib.load(CFG["real_image_path"]).dataobj).astype(np.float32)
    assert _img.shape == tuple(CFG["shape"]), f"image {_img.shape} != shape {CFG['shape']}"
    struct = make_struct_from_image(_img)
    print("struct: real image", CFG["real_image_path"])
elif CFG["sharp_edge"]:
    struct = make_struct_sharp(CFG["shape"], CFG["checker_contrast"], CFG["checker_period"])
    print("struct: sharp-edged disk")
else:
    struct = make_struct_analytic(CFG["shape"], CFG["checker_period"], CFG["checker_contrast"])
    print("struct: analytic checkerboard")

nu, phi, modulation = make_temporal(CFG["shape"], CFG["slice_block"],
                                    CFG["f_lo"], CFG["f_hi"], CFG["signal_amp"])
print(f"slice-block freqs: {sorted(set(np.round(nu,3)))} Hz")

## Motion — per-frame affine (voxel space) × `motion_scale`

`make_motion` returns 4×4 voxel-space matrices `M_f` and one constant shift `S`;
`ffs_nwarp` applies `T_f = M_f @ S`. `motion_scale` multiplies the whole trajectory
— set it to `2.0` to make a (real or synthetic) trajectory twice as large.

In [ ]:
def _rot_about_center(shape, ax, ay, az):
    nx, ny, nz = shape
    c = np.array([(nx - 1) / 2, (ny - 1) / 2, (nz - 1) / 2])
    Rx = np.array([[1, 0, 0], [0, np.cos(ax), -np.sin(ax)], [0, np.sin(ax), np.cos(ax)]])
    Ry = np.array([[np.cos(ay), 0, np.sin(ay)], [0, 1, 0], [-np.sin(ay), 0, np.cos(ay)]])
    Rz = np.array([[np.cos(az), -np.sin(az), 0], [np.sin(az), np.cos(az), 0], [0, 0, 1]])
    R = Rz @ Ry @ Rx
    M = np.eye(4); M[:3, :3] = R; M[:3, 3] = c - R @ c
    return M


def make_motion(nframes, shape, scale, through_plane, seed, amp_deg, amp_trans):
    """Synthetic drift + two step events. Returns (M_list, S), already scaled."""
    rng = np.random.default_rng(seed)
    t = np.arange(nframes)
    drift = lambda ph, sc: sc * np.sin(2 * np.pi * (t / nframes) + ph)
    az = np.deg2rad(drift(rng.uniform(0, 6), amp_deg) + 0.4 * rng.standard_normal(nframes))
    tx = drift(rng.uniform(0, 6), amp_trans) + 0.3 * rng.standard_normal(nframes)
    ty = drift(rng.uniform(0, 6), amp_trans) + 0.3 * rng.standard_normal(nframes)
    if through_plane:
        ax = np.deg2rad(1.5 * drift(rng.uniform(0, 6), amp_deg) + 0.4 * rng.standard_normal(nframes))
        ay = np.deg2rad(drift(rng.uniform(0, 6), amp_deg))
        tz = 1.2 * drift(rng.uniform(0, 6), amp_trans)
    else:
        ax = ay = tz = np.zeros(nframes)
    for spike in (nframes // 3, 2 * nframes // 3):
        az[spike:] += np.deg2rad(amp_deg); ty[spike:] += amp_trans
        if through_plane:
            tz[spike:] += 0.8 * amp_trans
    # Scale the whole trajectory (rotations and translations) about identity.
    ax, ay, az = ax * scale, ay * scale, az * scale
    tx, ty, tz = tx * scale, ty * scale, tz * scale
    M_list = []
    for f in range(nframes):
        M = _rot_about_center(shape, ax[f], ay[f], az[f])
        M[:3, 3] += np.array([tx[f], ty[f], tz[f]])
        M_list.append(M)
    S = np.eye(4)
    S[:3, 3] = np.array([1.5, -1.0, 0.7 if through_plane else 0.0]) * scale
    return M_list, S


def make_motion_oscillate(nframes, amp, scale):
    """Fast in-plane sweep: +/- amp voxels in x every frame. This is the regime
    where a fixed scanner voxel (tshift) and a fixed pose-j location (frozen joint)
    both see brain->air->brain across the temporal window, so both smear the wrong
    tissue in; only following the anatomy per frame recovers it."""
    mats = []
    for f in range(nframes):
        M = np.eye(4); M[0, 3] = scale * amp * (-1) ** f
        mats.append(M)
    return mats, np.eye(4)


def load_real_motion(path, scale):
    """Multi-row 12-col aff12 (voxel frame for this grid) -> [M_f], scaled about I.

    Adapt the loader to your motion source; the requirement is a per-frame 4x4
    voxel-space output->source matrix. Scaling here is linear about identity.
    """
    rows = np.loadtxt(path)
    rows = np.atleast_2d(rows)
    mats = []
    for r in rows:
        M = np.eye(4); M[:3, :] = r.reshape(3, 4)
        mats.append(np.eye(4) + scale * (M - np.eye(4)))
    return mats, np.eye(4)


if CFG["real_motion_path"]:
    M_list, S = load_real_motion(CFG["real_motion_path"], CFG["motion_scale"])
    print(f"motion: real {CFG['real_motion_path']} x{CFG['motion_scale']} ({len(M_list)} frames)")
elif CFG["motion_mode"] == "oscillate":
    M_list, S = make_motion_oscillate(CFG["nframes"], CFG["osc_amp"], CFG["motion_scale"])
    print(f"motion: oscillate +/-{CFG['osc_amp'] * CFG['motion_scale']} vox/frame in x")
else:
    M_list, S = make_motion(CFG["nframes"], CFG["shape"], CFG["motion_scale"],
                            CFG["through_plane"], CFG["motion_seed"],
                            CFG["amp_deg"], CFG["amp_trans"])
    print(f"motion: drift {'through-plane' if CFG['through_plane'] else 'in-plane'} "
          f"x{CFG['motion_scale']}")

# Quick trajectory readout (translation magnitude per frame).
_trans = np.array([np.linalg.norm((M @ S)[:3, 3]) for M in M_list])
print(f"|translation| range: {_trans.min():.2f} .. {_trans.max():.2f} vox")

## Slice timing — multiband, interleaved

In [ ]:
def make_slice_times(nz, tr, multiband, order):
    if nz % multiband != 0:
        raise ValueError(f"nz={nz} not divisible by multiband={multiband}")
    n_groups = nz // multiband
    if order == "interleaved":
        group_order = list(range(0, n_groups, 2)) + list(range(1, n_groups, 2))
    else:
        group_order = list(range(n_groups))
    times = np.zeros(nz)
    for pos, g in enumerate(group_order):
        for m in range(multiband):
            times[g + m * n_groups] = (pos / n_groups) * tr
    return times


if CFG["real_slicetiming"] is not None:
    slice_times = np.asarray(CFG["real_slicetiming"], float)
else:
    slice_times = make_slice_times(CFG["shape"][2], CFG["tr"],
                                   CFG["multiband"], CFG["slice_order"])
tzero = float(slice_times.mean())
print("slice times (s):", np.round(slice_times, 3))
print(f"tzero = {tzero:.3f} s")

## Forward corruption + (optional) noise

`acquired[f, v] = I(T_f^{-1} v,\ f\,TR + \Delta(v_z))` evaluated analytically, then
optional measurement noise from the repo's `generate_fmri_noise` (1/f + resp +
cardiac, independent per voxel) is added **at acquisition**. `ideal[j, u] =
I(u,\ j\,TR + t_0)` is the clean target every method is scored against.

In [ ]:
def _voxel_grid(shape):
    nx, ny, nz = shape
    ii, jj, kk = np.meshgrid(np.arange(nx), np.arange(ny), np.arange(nz), indexing="ij")
    g = np.stack([ii.reshape(-1), jj.reshape(-1), kk.reshape(-1), np.ones(ii.size)], 0)
    return g, (nx, ny, nz)


def generate_acquired(shape, struct, modulation, M_list, S, slice_times, tr, tzero):
    grid, (nx, ny, nz) = _voxel_grid(shape)
    kidx = grid[2].astype(int)
    out = np.zeros((nx, ny, nz, len(M_list)), np.float32)
    for f, M in enumerate(M_list):
        x = np.linalg.inv(M @ S) @ grid
        tau = f * tr + slice_times[kidx]
        out[..., f] = (struct(x) * modulation(x, tau)).reshape(nx, ny, nz)
    return out


def generate_ideal(shape, struct, modulation, nframes, tr, tzero):
    grid, (nx, ny, nz) = _voxel_grid(shape)
    base = struct(grid)
    out = np.zeros((nx, ny, nz, nframes), np.float32)
    for j in range(nframes):
        out[..., j] = (base * modulation(grid, j * tr + tzero)).reshape(nx, ny, nz)
    return out


def add_measurement_noise(vol4d, cfg):
    """Independent per-voxel 1/f + physiological noise, scaled to cfg['noise_sd']."""
    nx, ny, nz, nt = vol4d.shape
    torch.manual_seed(cfg["noise_seed"])
    n = generate_fmri_noise(
        tr=cfg["tr"], duration_s=nt * cfg["tr"], matrix_size=(1, nx * ny * nz),
        resp_strength=cfg["noise_resp_strength"],
        cardiac_strength=cfg["noise_cardiac_strength"],
        normalize=True, device=torch.device("cpu"),
    )                                        # (nt, 1, nvox), unit variance per voxel
    n = n.reshape(nt, nx, ny, nz).permute(1, 2, 3, 0).numpy()
    return vol4d + cfg["noise_sd"] * n.astype(np.float32)


acquired = generate_acquired(CFG["shape"], struct, modulation, M_list, S,
                             slice_times, CFG["tr"], tzero)
ideal = generate_ideal(CFG["shape"], struct, modulation, CFG["nframes"], CFG["tr"], tzero)
if CFG["noise_enabled"]:
    acquired = add_measurement_noise(acquired, CFG)
    print(f"added measurement noise, sd={CFG['noise_sd']}")
print("acquired", acquired.shape, " ideal", ideal.shape)

## Run the three corrections through the real CLIs

The known warp chain is written as AFNI `.aff12.1D` files. With a 1 mm LPI affine
(`diag(-1,-1,1,1)`) the file numbers *are* the output→source voxel matrix, so no
coordinate bookkeeping is needed. Output grids are pinned with `-master`.

In [ ]:
work = tempfile.mkdtemp()
aff = np.diag([-1.0, -1.0, 1.0, 1.0])
src = os.path.join(work, "acquired.nii.gz")
nib.save(nib.Nifti1Image(acquired, aff), src)

shift_p, motion_p, st_p = (os.path.join(work, n) for n in ("shift.1D", "motion.1D", "st.1D"))
with open(shift_p, "w") as f:
    f.write(" ".join(f"{v:.8f}" for v in S[:3, :].reshape(-1)) + "\n")
with open(motion_p, "w") as f:
    for M in M_list:
        f.write(" ".join(f"{v:.8f}" for v in M[:3, :].reshape(-1)) + "\n")
np.savetxt(st_p, slice_times)
chain = f"{shift_p} {motion_p}"
dev, tr = CFG["device"], str(CFG["tr"])

paths = {k: os.path.join(work, f"{k}.nii.gz")
         for k in ("joint", "follow", "moco", "tshift", "tshiftmoco")}
st_args = ["-tpattern", st_p, "-TR", tr, "-tzero", str(tzero),
           "-tinterp", CFG["tinterp"], "-interp", CFG["interp"], "-master", src,
           "-device", dev, "-verb", "0"]

# joint (frozen pose)
nwarp_main(["-source", src, "-nwarp", chain, "-prefix", paths["joint"], *st_args])
# joint (tissue-following): same call + -tfollow
nwarp_main(["-source", src, "-nwarp", chain, "-prefix", paths["follow"], "-tfollow", *st_args])
# motion-only
nwarp_main(["-source", src, "-nwarp", chain, "-prefix", paths["moco"],
            "-interp", CFG["interp"], "-master", src, "-device", dev, "-verb", "0"])
# tshift-then-motion (correct two-step)
slicetime_main(["-input", src, "-prefix", paths["tshift"], "-tpattern", st_p,
                "-TR", tr, "-tzero", str(tzero), "-wsinc5", "-device", dev])
nwarp_main(["-source", paths["tshift"], "-nwarp", chain, "-prefix", paths["tshiftmoco"],
            "-interp", CFG["interp"], "-master", src, "-device", dev, "-verb", "0"])

rec = {
    "follow": np.asarray(nib.load(paths["follow"]).dataobj).astype(np.float32),
    "joint": np.asarray(nib.load(paths["joint"]).dataobj).astype(np.float32),
    "tshift": np.asarray(nib.load(paths["tshiftmoco"]).dataobj).astype(np.float32),
    "moco": np.asarray(nib.load(paths["moco"]).dataobj).astype(np.float32),
}
print("done:", {k: v.shape for k, v in rec.items()})

## Metrics — recovery vs the clean truth

In [ ]:
from scipy.ndimage import binary_erosion

def interior_mask(shape, struct, erode=3, z_erode=0):
    grid, (nx, ny, nz) = _voxel_grid(shape)
    m = struct(grid).reshape(nx, ny, nz) > 0.2
    fp = np.zeros((3, 3, 1), bool); fp[:, 1, 0] = fp[1, :, 0] = True
    m = binary_erosion(m, structure=fp, iterations=erode)
    if z_erode:
        m[:, :, :z_erode] = False; m[:, :, nz - z_erode:] = False
    return m


def temporal_corr(a, b, mask):
    A, B = a[mask], b[mask]
    A = A - A.mean(1, keepdims=True); B = B - B.mean(1, keepdims=True)
    return (A * B).sum(1) / (np.sqrt((A**2).sum(1) * (B**2).sum(1)) + 1e-12)


def amplitude_slope(rec, ideal, mask):
    R, I = rec[mask], ideal[mask]
    R = R - R.mean(1, keepdims=True); I = I - I.mean(1, keepdims=True)
    return (R * I).sum(1) / ((I**2).sum(1) + 1e-12)


mask = interior_mask(CFG["shape"], struct, z_erode=2 if CFG["through_plane"] else 0)
grid, sh = _voxel_grid(CFG["shape"])
kmap = grid[2].reshape(sh).astype(int)
freq = nu[kmap[mask]]
corr = {m: temporal_corr(v, ideal, mask) for m, v in rec.items()}
ampl = {m: amplitude_slope(v, ideal, mask) for m, v in rec.items()}

methods = ["follow", "joint", "tshift", "moco"]
hdr = {"follow": "follow", "joint": "joint", "tshift": "tshift", "moco": "moco"}
edges = np.quantile(freq, [0, 1/3, 2/3, 1.0])
mtag = ("oscillate" if CFG["motion_mode"] == "oscillate"
        else ("through-plane" if CFG["through_plane"] else "in-plane"))
print(f"{int(mask.sum())} interior voxels | {mtag} x{CFG['motion_scale']} | "
      f"noise={'on' if CFG['noise_enabled'] else 'off'}   (corr with truth)\n")
row = lambda name, vals: print(f"{name:<18}| " + " ".join(f"{v:>7.3f}" for v in vals))
print(f"{'band':<18}| " + " ".join(f"{hdr[m]:>7}" for m in methods))
print("-" * 52)
for lo, hi, name in [(edges[0], edges[1], 'slow'), (edges[1], edges[2], 'mid'),
                     (edges[2], edges[3] + 1e-9, 'fast')]:
    s = (freq >= lo) & (freq < hi)
    row(f"{name} {lo:.2f}-{hi:.2f} Hz", [corr[m][s].mean() for m in methods])
print("-" * 52)
row("OVERALL", [corr[m].mean() for m in methods])
row("amplitude", [ampl[m].mean() for m in methods])

## Figures

Colours: **joint** (blue), **tshift→moco** (green), **motion-only** (red),
**truth** (black).

In [ ]:
STYLE = {"follow": ("C4", "D-", "joint (tissue-following)"),
         "joint": ("C0", "o-", "joint (frozen pose)"),
         "tshift": ("C2", "^-", "tshift-then-motion"),
         "moco": ("C3", "s--", "motion-only")}

# ---- Fig 1: spatial montage (corruption vs correction) at a mid slice --------
kz = CFG["shape"][2] // 2
frames = [CFG["nframes"] // 4, CFG["nframes"] // 2, 3 * CFG["nframes"] // 4]
rows = [("acquired (corrupted)", acquired), ("joint (frozen)", rec["joint"]),
        ("joint (following)", rec["follow"]), ("ideal (truth)", ideal)]
fig, axes = plt.subplots(len(rows), len(frames), figsize=(3 * len(frames), 3 * len(rows)))
vmax = np.percentile(ideal[..., kz, :], 99)
for ri, (lab, vol) in enumerate(rows):
    for ci, fr in enumerate(frames):
        ax = axes[ri, ci]
        ax.imshow(vol[:, :, kz, fr].T, cmap="gray", vmin=0, vmax=vmax, origin="lower")
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if ri == 0:
            ax.set_title(f"frame {fr}")
        if ci == 0:
            ax.set_ylabel(lab, fontsize=11)
fig.suptitle(f"slice z={kz}: acquired (moving) -> frozen joint (edge ghost) -> "
             f"following (clean) -> truth", y=1.0, fontsize=11)
fig.tight_layout(); plt.show()

In [ ]:
# ---- Fig 2: recovered timeseries at a legible mid-high-frequency voxel -------
kdemo = min(range(len(nu)), key=lambda k: abs(nu[k] - 0.29))
idx = np.argwhere(mask & (kmap == kdemo))
vx = tuple(int(c) for c in idx[len(idx) // 2])

# row of this voxel within the masked arrays (C-order), for a legend r-value
vlin = int(np.ravel_multi_index(vx, CFG["shape"]))
mrow = int(np.flatnonzero(np.flatnonzero(mask.reshape(-1)) == vlin)[0])

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ideal[vx], "k-", lw=2.4, label="ideal (truth)")
for m, (c, _, lab) in STYLE.items():
    ax.plot(rec[m][vx], c + ".-", lw=1.3, alpha=0.9, label=f"{lab}  (r={corr[m][mrow]:.3f})")
ax.set_title(f"voxel {vx}  |  slice frequency {nu[kdemo]:.2f} Hz")
ax.set_xlabel("frame"); ax.set_ylabel("signal"); ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

In [ ]:
# ---- Fig 3: recovery & amplitude vs signal frequency ------------------------
# Signals are block-piecewise-constant, so aggregate at the actual block freqs.
ufreq = np.unique(freq)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for metric, ax, ylab, title in [(corr, axes[0], "corr with truth", "signal recovery"),
                                 (ampl, axes[1], "amplitude ratio", "amplitude recovery")]:
    for m, (c, mk, lab) in STYLE.items():
        ys = [metric[m][freq == uf].mean() for uf in ufreq]
        ax.plot(ufreq, ys, c + mk, lw=1.6, ms=7, alpha=0.85, label=lab)
    ax.axvline(NYQUIST, ls=":", c="k", alpha=0.4)
    ax.set_ylim(0, 1.05); ax.set_xlabel("slice signal frequency (Hz)")
    ax.set_ylabel(ylab); ax.set_title(title); ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

In [ ]:
# ---- Fig 4: temporal RMS-error maps on a fast slice (where does each method fail?)
kfast = int(np.argmax(nu))                       # fastest signal block
mslice = np.zeros(CFG["shape"], bool); mslice[:, :, kfast] = mask[:, :, kfast]

def rms_err(rec):
    e = np.sqrt(((rec - ideal) ** 2).mean(-1))   # (nx,ny,nz)
    e[~mslice] = np.nan
    return e[:, :, kfast]

emaps = {m: rms_err(rec[m]) for m in STYLE}
vmax = np.nanpercentile(np.stack(list(emaps.values())), 99)
fig, axes = plt.subplots(1, len(STYLE), figsize=(3.3 * len(STYLE), 4.4))
for ax, (m, (_, _, lab)) in zip(axes, STYLE.items()):
    im = ax.imshow(emaps[m].T, origin="lower", cmap="magma", vmin=0, vmax=vmax)
    ax.set_title(f"{lab}\nmean RMS {np.nanmean(emaps[m]):.3f}")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02, label="temporal RMS error")
fig.suptitle(f"residual error on the fastest slice (z={kfast}, {nu[kfast]:.2f} Hz)", y=1.02)
plt.show()

## Reading the results, and how to push on it

- **In-plane, no noise** — joint ≈ tshift→moco ≈ 0.99, both far above motion-only,
  the gap widening with frequency. This validates the joint path: it reproduces
  the established two-step to ~1% while clearly beating no-timing correction. In
  apply-only mode (motion already known) joint and the two-step *should* agree — a
  sample's acquisition time is a property of the scanner slice that both assign
  correctly.
- **Through-plane** (`through_plane=True`) — every method's ceiling drops, because
  tissue crosses slice boundaries and any resample must interpolate a fixed
  location's timeseries across frames. Joint's one-pose-per-frame slow-motion
  assumption gives up a little here; this is where *joint estimation* (not just
  application) would pay off.
- **Noise on** (`noise_enabled=True`) — adds the repo's 1/f + respiratory +
  cardiac noise at acquisition. Both slice-timing-aware methods degrade gracefully
  and stay ahead of motion-only; joint and tshift-then-motion keep tracking each
  other. That's expected: `3dTshift` is a *purely temporal* resample (it adds no
  spatial blur), so the correct two-step already uses only one spatial
  interpolation — joint has no "fewer resamples" edge over it here. Joint's real
  advantage is elsewhere: joint *estimation* of motion+timing, and avoiding the
  **wrong** motion-*then*-tshift ordering (tshift on motion-corrected data assumes
  a slice association the motion already broke). This notebook deliberately
  compares against the *correct* two-step, which is the honest bar.
- **Tissue-following** (`-tfollow`, the `follow` method) is the payoff. Under slow
  drift it matches the frozen joint (motion within a temporal window is sub-voxel,
  so the pose barely changes). Switch `motion_mode="oscillate"` (fast ±`osc_amp`
  vox/frame, `sharp_edge=True` for a brain/air boundary) and it pulls decisively
  ahead: the frozen joint and tshift both smear the wrong tissue across the
  temporal window (a fixed pose-j location / fixed scanner voxel sees brain→air→
  brain), while following samples each tap at its own frame's pose and stays on the
  anatomy. That's the regime real edges live in — a voxel blinking in and out of
  the brain as the head moves.
- **Scale the motion** with `motion_scale` (e.g. `2.0`) to stress the slow-motion
  assumption; **drop in real data** via `real_image_path` / `real_motion_path` /
  `real_slicetiming` and keep the synthetic signal riding on top.